In [3]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from scipy.signal import find_peaks, savgol_filter

# === Core Functions ===

def create_global_spectrum(X, mz_axis):
    if hasattr(X, "toarray"):  # for sparse matrix
        X = X.toarray()
    global_spectrum = np.sum(X, axis=0)
    return mz_axis, global_spectrum

def detect_peaks(mz_axis, global_spectrum, 
                 window_length=9, polyorder=2,
                 min_prominence=0.02, top_n=None,
                 da=0.01, add_da=0.01,
                 save_path="detected_peaks.txt"):
    
    
    # Step 0: Savitzky-Golay smoothing
    smoothed = savgol_filter(global_spectrum, window_length=window_length, polyorder=polyorder)
    
    # Step 1: Peak detection
    peaks, _ = find_peaks(smoothed, prominence=min_prominence)
    print(f"Initial detected peaks: {len(peaks)}")

    mz_peaks = mz_axis[peaks]
    intensities = smoothed[peaks]

    # Step 2: Sort peaks by intensity descending
    sorted_indices = np.argsort(intensities)[::-1]
    mz_peaks_sorted = mz_peaks[sorted_indices]
    intensities_sorted = intensities[sorted_indices]

    # Step 3: Filter out peaks that are too close (within ±da of a stronger one)
    final_mz = []
    final_intensity = []
    for i, mz in enumerate(mz_peaks_sorted):
        if not any(np.abs(mz - accepted_mz) <= da for accepted_mz in final_mz):
            final_mz.append(mz)
            final_intensity.append(intensities_sorted[i])

    final_mz = np.array(final_mz)
    final_intensity = np.array(final_intensity)

    print(f"Filtered peaks (unique by ±{da} Da): {len(final_mz)}")

    # Step 4: Optional: 2*Top-N filtering after proximity filtering
    if top_n is not None and len(final_mz) > 2*top_n:
        top_indices = np.argsort(final_intensity)[-2*top_n:][::-1]
        final_mz = final_mz[top_indices]
        final_intensity = final_intensity[top_indices]

    # Step 5: Add intensity of all m/z points in ±da range from global_spectrum
    for i, mz in enumerate(final_mz):
        in_range = np.abs(mz_axis - mz) <= add_da
        # Exclude the exact center peak to avoid doubling
        in_range &= ~np.isclose(mz_axis, mz, atol=1e-8)
        added_intensity = np.sum(smoothed[in_range])
        final_intensity[i] += added_intensity

    # Step 6: Final sort by updated intensity and select top N
    if top_n is not None and len(final_mz) > top_n:
        final_sorted_indices = np.argsort(final_intensity)[-top_n:][::-1]
        final_mz = final_mz[final_sorted_indices]
        final_intensity = final_intensity[final_sorted_indices]

    # Step 6: Save to file
    np.savetxt(save_path, np.column_stack((final_mz, final_intensity)),
               fmt="%.6f", header="m/z\tintensity", delimiter="\t")

    return final_mz, final_intensity



In [4]:
# === Parameters ===
input_file = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/PeakDetection/a.h5ad"
#output_file = "/Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/a_hdi_style_1000_0.04_100.h5ad"

# === Load and Process ===
adata = sc.read_h5ad(input_file)
print(f"Loaded AnnData from: {input_file}")

X = adata.X
X = X.toarray() if hasattr(X, "toarray") else X
row_sums = X.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1  # avoid division by zero
X_norm = X / row_sums

mz_axis, global_spec = create_global_spectrum(X, adata.var_names.astype(float).values)
#mz_axis, global_spec = create_global_spectrum_max(adata.X, adata.var_names.astype(float).values)
da_list = [0.1, 0.05, 0.024, 0.02, 0.01]
add_da_list = [0.0001, 0.001, 0.01, 0.02, 0.024, 0.03, 0.04, 0.05]
window_length_list = [3, 5, 7, 9, 11, 13, 15, 17, 19, 21]
polyorder_list = [0, 1, 2, 3]

# === Peak Detection and Saving ===
for window_length in window_length_list:
    for polyorder in polyorder_list:
        if window_length > polyorder:
            for da in da_list:
                for add_da in add_da_list:
                    if add_da < da:
                        peak_mz, peak_intensities = detect_peaks(mz_axis, global_spec, 
                                                                window_length=window_length, polyorder=polyorder,
                                                                min_prominence=0.00001, top_n=1000,
                                                                da=da,add_da=add_da,
                                                                save_path=f"txt_files/first_top_peaks_00001_{da}da_add_{add_da}da_{window_length}w{polyorder}p_savit_glob_1000top_sum.txt")
                        print(f'Number of peaks: {len(peak_mz)}')
                        print(f"Saved peaks to: txt_files/first_top_peaks_00001_{da}da_add_{add_da}da_{window_length}w{polyorder}p_savit_glob_1000top_sum.txt")
                

/opt/anaconda3/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Loaded AnnData from: /Users/amin/Desktop/PhD_Thesis/Code/IMZML Tools/PeakDetection/a.h5ad
Initial detected peaks: 16802
Filtered peaks (unique by ±0.1 Da): 8048
Number of peaks: 1000
Saved peaks to: txt_files/first_top_peaks_00001_0.1da_add_0.0001da_3w0p_savit_glob_1000top_sum.txt
Initial detected peaks: 16802
Filtered peaks (unique by ±0.1 Da): 8048
Number of peaks: 1000
Saved peaks to: txt_files/first_top_peaks_00001_0.1da_add_0.001da_3w0p_savit_glob_1000top_sum.txt
Initial detected peaks: 16802
Filtered peaks (unique by ±0.1 Da): 8048
Number of peaks: 1000
Saved peaks to: txt_files/first_top_peaks_00001_0.1da_add_0.01da_3w0p_savit_glob_1000top_sum.txt
Initial detected peaks: 16802
Filtered peaks (unique by ±0.1 Da): 8048
Number of peaks: 1000
Saved peaks to: txt_files/first_top_peaks_00001_0.1da_add_0.02da_3w0p_savit_glob_1000top_sum.txt
Initial detected peaks: 16802
Filtered peaks (unique by ±0.1 Da): 8048
Number of peaks: 1000
Saved peaks to: txt_files/first_top_peaks_00001_0.1da_